In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 129.3 MB/s eta 0:00:00


In [ ]:
import os, re, json, string, gc
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm

MODEL_ID = "NousResearch/Meta-Llama-3.1-8B-Instruct"
SEED, MAX_NEW = 42, 48
OUT_DIR = "./p8_steer"; os.makedirs(OUT_DIR, exist_ok=True)

DRIVE_DIR = '/content/drive/MyDrive/Grad_share/data_share/데이터셋 모음'   # p8s_pairs.csv 위치
SAVE_DIR  = "/content/drive/MyDrive/Grad_share/data_share/P9"; os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
def normalize(s):
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    return " ".join(s.split())

def match_gold(text, golds):
    t = normalize(text)
    for g in golds:
        if not str(g).strip(): continue
        if re.search(r"\b"+re.escape(normalize(g))+r"\b", t): return True
    return False

REFUSAL_PAT = re.compile(
    r"(i (don'?t|do not) know|couldn'?t find|cannot|unable to|no information|not sure)",
    re.IGNORECASE)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
model.eval()

N_LAYERS = model.config.num_hidden_layers   # 32
D_MODEL  = model.config.hidden_size         # 4096
print(f"로드 완료: {N_LAYERS} layers x {D_MODEL} dim")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

로드 완료: 32 layers x 4096 dim


In [ ]:
import numpy as np, torch, json, re, string, pandas as pd
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

DATA = "/content/drive/MyDrive/Grad_share/data_share/데이터셋 모음"
P9   = "/content/drive/MyDrive/Grad_share/data_share/P9"

pairs9 = pd.read_csv(f"{P9}/p8s_behavior.csv")          # ← behavior 포함본
HID9   = np.load(f"{P9}/hid_p8s.npy").astype(np.float32)
ids    = np.load(f"{P9}/hid_p8s_ids.npy", allow_pickle=True)

print("pairs9:", pairs9.shape, "| HID9:", HID9.shape, "| ids:", ids.shape)
print("컬럼:", [c for c in ['behavior','question','ctx_text','gold','kside','ctx_label'] if c in pairs9.columns])
print("behavior 분포:\n", pairs9.behavior.value_counts().to_dict() if 'behavior' in pairs9 else "behavior 없음")
assert HID9.shape[0]==len(pairs9) and HID9.shape[1]==32*4096, (HID9.shape, len(pairs9))

def layer(H, L): return H[:, (L-1)*4096:L*4096]
def diff_of_means(H, ma, mb):
    W=np.zeros((32,4096),np.float32)
    for L in range(1,33):
        HL=layer(H,L); d=HL[ma].mean(0)-HL[mb].mean(0)
        W[L-1]=d/(np.linalg.norm(d)+1e-8)
    return W
def em(ans, gold):
    return int(match_gold(ans, json.loads(gold) if str(gold).startswith("[") else [gold]))
def collapsed(t):
    w=t.split()
    if len(w)>=8 and len(set(w))/len(w)<0.3: return True
    return re.search(r"(.{2,10})\1{3,}", t) is not None
def cluster_bootstrap(base, cur, groups, n_boot=2000, seed=0):
    rng=np.random.default_rng(seed); uq=np.unique(groups)
    idx={q:np.where(groups==q)[0] for q in uq}; nets=[]
    for _ in range(n_boot):
        rows=np.concatenate([idx[q] for q in rng.choice(uq,len(uq),replace=True)])
        b,c=base[rows],cur[rows]
        nets.append(((b==0)&(c==1)).sum()-((b==1)&(c==0)).sum())
    return dict(net_mean=np.mean(nets), net_CI=(np.percentile(nets,2.5),np.percentile(nets,97.5)))

pairs9: (1224, 19) | HID9: (1224, 131072) | ids: (1224,)
컬럼: ['behavior', 'question', 'ctx_text', 'gold', 'kside', 'ctx_label']
behavior 분포:
 {'accept': 880, 'retain': 201, 'other': 142, 'abstain': 1}


In [ ]:
print(list(pairs9.columns))

['question', 'gold', 'prop', 'kside', 'knows', 'A_cb', 'ctx_id', 'ctx_title', 's_wiki_title', 'orig_gold', 'ctx_label', 'ctx_text', 'answer_surface', 'ans_start', 'ans_end', 'wrong_answer']


In [ ]:
import glob
print([f.split('/')[-1] for f in glob.glob(f"{P9}/*") + glob.glob(f"{DATA}/*")])

['hid_p8s.npy', 'hid_p8s_ids.npy', 'p8s_behavior.csv', 'w_directions.npz', 'steer_sweep.csv', 'acc_detail.npz', 'retain_detail.npz', 'retain_ext16.npz', 'retain48.npz', 'acc_front.npz', 'w_behavior.npy', 'w_behavior_sf.npy', 'w_behavior_na.npy', 'w_behavior_na2.npy', 'hid_sv_p9prompt.npy', 'hid_sv_p9prompt_ids.npy', 'sv_layer_ckpt.npz', 'steer_na2_ckpt.npz', 'retain_oof.npz', 'acc48.npz', 'figs', 'retain_na_m3.npy', 'acc_split.npz', 'retain_front_oof.npz', 'accept_push.npz', 'retain_ext.npz', 'p8_pairs.csv', 'p8s_pairs.csv']


In [ ]:
key = 'ctx_id' if 'ctx_id' in pairs9.columns else pairs9.columns[0]
print("정렬 일치?", np.array_equal(pairs9[key].values.astype(str), ids.astype(str)))

정렬 일치? False


In [ ]:
print("ids 예시:", ids[:3], "| dtype:", ids.dtype)
print("ids 중복?", len(ids) != len(set(map(str, ids))), "| 고유수:", len(set(map(str,ids))))
for c in pairs9.columns:
    v = pairs9[c].astype(str).values
    same = np.array_equal(sorted(v), sorted(map(str, ids)))   # 순서 무시 집합 일치
    if same: print("★ 집합 일치 컬럼:", c, "| 고유?", pairs9[c].is_unique)

ids 예시: ['What genre is Kanoon?' 'Who is the author of Free to Choose?'
 'What is Prague the capital of?'] | dtype: object
ids 중복? True | 고유수: 612
★ 집합 일치 컬럼: question | 고유? False


In [ ]:
# question 등장 순서가 pairs9와 같은지 (같으면 이미 정렬됨)
print("등장순서 일치?", np.array_equal(pairs9.question.values.astype(str), ids.astype(str)))

등장순서 일치? True


In [ ]:
def build_prompt_str(row):
    return tok.apply_chat_template(
        [{"role":"user","content":f"{row.ctx_text}\n\nQuestion: {row.question}\nAnswer:"}],
        add_generation_prompt=True, tokenize=False)

# σ_proj: fold-train pooled w 방향으로 사영한 std (평가행 누수 없음)
ACC, RET = "accept", "retain"
allm = pairs9.behavior.isin([ACC,RET]).values
sig_proj = np.zeros((len(pairs9),32), np.float32)
for tr,te in GroupKFold(5).split(np.arange(len(pairs9)), groups=pairs9.question.values):
    gi  = np.intersect1d(tr, np.where(allm)[0])
    lab = pairs9.behavior.values[gi]
    Wtr = diff_of_means(HID9[gi], lab==ACC, lab==RET)
    for L in range(1,33):
        sig_proj[te, L-1] = (layer(HID9[gi],L) @ Wtr[L-1]).std()
sigma = np.median(sig_proj, axis=0).astype(np.float32)
sig_t = torch.tensor(sigma, device=model.device, dtype=torch.bfloat16)
w_t   = torch.zeros((32,4096), device=model.device, dtype=torch.bfloat16)

_hooks, _cfg = [], {"coef":0.0, "layers":[]}
def make_hook(Li):
    def hook(mod, inp, out):
        if _cfg["coef"]==0.0 or Li not in _cfg["layers"]: return out
        h = out[0] if isinstance(out,tuple) else out
        h = h + _cfg["coef"]*sig_t[Li]*w_t[Li].to(h.dtype)
        return (h,)+out[1:] if isinstance(out,tuple) else h
    return hook
def attach():
    for i in range(32): _hooks.append(model.model.layers[i].register_forward_hook(make_hook(i)))
def detach():
    for h in _hooks: h.remove()
    _hooks.clear()

# 진단
kpcm = pairs9[(pairs9.kside=="K+")&(pairs9.ctx_label=="C-")].index.values
sig_coord = np.array([layer(HID9[kpcm],L).std() for L in range(1,33)], np.float32)
print("σ_proj[7,13,15,17]:", sigma[[7,13,15,17]].round(3), " (≈0.9~1.5 기대)")
print("배율 r[13,15,17]:", (sigma/sig_coord)[[13,15,17]].round(1), " (~9배 기대)")
print("σ_proj 행변동(15층): std", sig_proj[:,14].std().round(3), "/ 중앙", np.median(sig_proj[:,14]).round(3))

σ_proj[7,13,15,17]: [0.27  0.77  1.194 1.876]  (≈0.9~1.5 기대)
배율 r[13,15,17]: [ 7.3  9.  10.9]  (~9배 기대)
σ_proj 행변동(15층): std 0.012 / 중앙 0.857


In [ ]:
W_oof = np.zeros((len(pairs9),32,4096), np.float32)
for tr,te in GroupKFold(5).split(np.arange(len(pairs9)), groups=pairs9.question.values):
    gi  = np.intersect1d(tr, np.where(allm)[0])
    lab = pairs9.behavior.values[gi]
    W_oof[te] = diff_of_means(HID9[gi], lab==ACC, lab==RET)
print("W_oof:", W_oof.shape, "| norm[13,15,17]:", np.linalg.norm(W_oof[kpcm[0]],axis=1)[[13,15,17]].round(2))

W_oof: (1224, 32, 4096) | norm[13,15,17]: [1. 1. 1.]


In [ ]:
BLOCKS = {"front":[8,9,10,11,12], "mid1":[11,12,13,14,15],
          "signal":[14,15,16,17,18], "back":[17,18,19,20,21]}
ACC_C = [0.3, 0.5, 1.0, 1.5]      # 투영 눈금 (좌표 ~3,~5,~9,~13.5)
RET_C = [-0.3, -0.5, -1.0, -1.5]
acc_idx = pairs9[(pairs9.kside=="K−")&(pairs9.ctx_label=="C+")].index.values
ret_idx = pairs9[(pairs9.kside=="K+")&(pairs9.ctx_label=="C-")].index.values
q_acc = pairs9.loc[acc_idx,"question"].values
q_ret = pairs9.loc[ret_idx,"question"].values
print("accept셀", len(acc_idx), "| retain셀", len(ret_idx))

@torch.no_grad()
def gen_em(i, coef, block):
    _cfg["coef"], _cfg["layers"] = coef, block
    r = pairs9.loc[i]
    w_t.copy_(torch.tensor(W_oof[i], device=model.device, dtype=torch.bfloat16))
    enc = tok(build_prompt_str(r), return_tensors="pt", add_special_tokens=False).to(model.device)
    L = enc["input_ids"].shape[1]
    g = model.generate(**enc, max_new_tokens=48, do_sample=False, pad_token_id=tok.pad_token_id)
    a = tok.decode(g[0,L:], skip_special_tokens=True).strip()
    return em(a, r.gold), collapsed(a)

for lyr in model.model.layers: lyr._forward_hooks.clear()
_hooks.clear(); attach()

def sweep_dir(idxs, coefs, qs, tag):
    base = np.array([gen_em(i,0.0,[14,15,16,17,18])[0] for i in tqdm(idxs, desc=f"{tag}-base", leave=False)])
    print(f"\n=== {tag} (baseline EM {base.mean():.3f}, 틀린 것 {(base==0).sum()}) ===")
    best={}
    for bn, blk in BLOCKS.items():
        for c in coefs:
            res=[gen_em(i,c,blk) for i in tqdm(idxs, desc=f"{tag}-{bn}{c}", leave=False)]
            cur=np.array([x[0] for x in res]); col=np.mean([x[1] for x in res])
            rec=((base==0)&(cur==1)).sum(); brk=((base==1)&(cur==0)).sum()
            ci=cluster_bootstrap(base,cur,qs)["net_CI"]; net=rec-brk
            if bn not in best or net>best[bn][1]: best[bn]=(c,net,tuple(map(int,ci)),col)
            print(f"  {bn:7} c{c:+.1f}: 순증 {net:+4} CI{tuple(map(int,ci))} 붕괴 {col:.2f}")
        b=best[bn]; print(f"    → {bn} 최적 c{b[0]:+.1f} 순증{b[1]:+} 붕괴{b[3]:.2f}")
    return base, best

acc_base, acc_best = sweep_dir(acc_idx, ACC_C, q_acc, "accept")
ret_base, ret_best = sweep_dir(ret_idx, RET_C, q_ret, "retain")
detach()

np.savez(f"{P9}/block_sweep_proj.npz",
         acc_best=np.array([(k,)+v[:2] for k,v in acc_best.items()], dtype=object),
         ret_best=np.array([(k,)+v[:2] for k,v in ret_best.items()], dtype=object))
print("\n저장:", f"{P9}/block_sweep_proj.npz")

accept셀 234 | retain셀 378


accept-base:   0%|          | 0/234 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



=== accept (baseline EM 0.868, 틀린 것 31) ===


accept-front0.3:   0%|          | 0/234 [00:00<?, ?it/s]

...

  back    c-0.5: 순증  +19 CI(0, 38) 붕괴 0.00


retain-back-1.0:   0%|          | 0/378 [00:00<?, ?it/s]

  back    c-1.0: 순증  -44 CI(-64, -23) 붕괴 0.00


retain-back-1.5:   0%|          | 0/378 [00:00<?, ?it/s]

  back    c-1.5: 순증  -93 CI(-114, -72) 붕괴 0.01
    → back 최적 c-0.3 순증+27 붕괴0.00

저장: /content/drive/MyDrive/Grad_share/data_share/P9/block_sweep_proj.npz


In [ ]:
!nvidia-smi --query-gpu=name,utilization.gpu,memory.used --format=csv

name, utilization.gpu [%], memory.used [MiB]
NVIDIA A100-SXM4-40GB, 0 %, 15970 MiB


In [ ]:
import time
_cfg["coef"]=0.0; _cfg["layers"]=[14,15,16,17,18]
for lyr in model.model.layers: lyr._forward_hooks.clear()
attach()
i=int(ret_idx[0]); r=pairs9.loc[i]
w_t.copy_(torch.tensor(W_oof[i], device=model.device, dtype=torch.bfloat16))
enc=tok(build_prompt_str(r), return_tensors="pt", add_special_tokens=False).to(model.device)
t=time.time()
g=model.generate(**enc, max_new_tokens=48, do_sample=False, pad_token_id=tok.pad_token_id)
dt=time.time()-t
detach()
print(f"1건 {dt:.2f}s | 생성토큰 {g.shape[1]-enc['input_ids'].shape[1]}")

1건 0.08s | 생성토큰 2


In [ ]:
import time
# 1) W_oof GPU 사전적재 (retain 행만)
W_gpu = {int(i): torch.tensor(W_oof[i], device=model.device, dtype=torch.bfloat16)
         for i in ret_idx}
# 2) 프롬프트 사전 렌더
prompt_cache = {int(i): build_prompt_str(pairs9.loc[i]) for i in ret_idx}

BLOCKS_R = {"mid1":[11,12,13,14,15], "signal":[14,15,16,17,18], "back":[17,18,19,20,21]}
RET_C = [-0.3,-0.5,-0.7,-1.0]

# 3) 선택적 훅: 개입 층에만 부착 (전 32층 → 5층)
def attach_sel(layers):
    detach()
    for i in layers:
        _hooks.append(model.model.layers[i].register_forward_hook(make_hook(i)))

_cur = [None]
@torch.no_grad()
def gen_txt(i, coef, block):
    if block != _cur[0]:
        attach_sel(block); _cur[0] = block      # 블록 바뀔 때만 재부착
    _cfg["coef"], _cfg["layers"] = coef, block
    w_t.copy_(W_gpu[i])
    enc = tok(prompt_cache[i], return_tensors="pt", add_special_tokens=False).to(model.device)
    L = enc["input_ids"].shape[1]
    g = model.generate(**enc, max_new_tokens=48, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(g[0,L:], skip_special_tokens=True).strip()

for lyr in model.model.layers: lyr._forward_hooks.clear()
_hooks.clear()

rows=[]; t0=time.time()
for i in tqdm(ret_idx, desc="base"):
    rows.append(("retain",int(i),"base",0.0, gen_txt(i,0.0,[14,15,16,17,18])))
for bn,blk in BLOCKS_R.items():
    for c in RET_C:
        for i in tqdm(ret_idx, desc=f"{bn}{c}", leave=False):
            rows.append(("retain",int(i),bn,c, gen_txt(i,c,blk)))
detach()
print(f"총 {time.time()-t0:.0f}s, {len(rows)}건")

gen_df = pd.DataFrame(rows, columns=["dir","idx","block","coef","text"])
gen_df.to_csv(f"{P9}/retain_sweep_gen.csv", index=False)
print("저장:", f"{P9}/retain_sweep_gen.csv")

base:   0%|          | 0/378 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import time
i=int(ret_idx[0])
# a) 프롬프트 캐시 조회 + 토크나이즈
t=time.time()
for _ in range(20):
    enc=tok(prompt_cache[i], return_tensors="pt", add_special_tokens=False).to(model.device)
print("tok+to(device) 20회:", round(time.time()-t,2),"s")
# b) 순수 생성 (훅 없음)
detach()
t=time.time()
for _ in range(20):
    g=model.generate(**enc, max_new_tokens=48, do_sample=False, pad_token_id=tok.pad_token_id)
print("generate 20회:", round(time.time()-t,2),"s | 마지막 생성토큰", g.shape[1]-enc['input_ids'].shape[1])

tok+to(device) 20회: 0.02 s
generate 20회: 1.66 s | 마지막 생성토큰 2


In [ ]:
import time
t=time.time()
W_gpu = {int(i): torch.tensor(W_oof[i], device=model.device, dtype=torch.bfloat16) for i in ret_idx}
print("W_gpu 생성:", round(time.time()-t,1),"s")

t=time.time()
prompt_cache = {int(i): build_prompt_str(pairs9.loc[i]) for i in ret_idx}
print("prompt_cache:", round(time.time()-t,1),"s")

W_gpu 생성: 0.1 s
prompt_cache: 0.1 s


In [ ]:
import time
detach(); _cur=[None]
t=time.time()
for k,i in enumerate(ret_idx[:20]):
    _=gen_txt(int(i),0.0,[14,15,16,17,18])
dt=time.time()-t
print(f"base 20건: {dt:.2f}s → 건당 {dt/20:.3f}s → 378건 예상 {dt/20*378:.0f}s")

base 20건: 20.97s → 건당 1.049s → 378건 예상 396s


In [ ]:
import time
BLOCKS_R = {"mid1":[11,12,13,14,15], "signal":[14,15,16,17,18], "back":[17,18,19,20,21]}
RET_C = [-0.3,-0.5,-0.7,-1.0]

@torch.no_grad()
def gen_txt(i):                         # 훅 로직 없음 — 부착은 밖에서
    _cfg["coef"] = _cfg["coef"]         # (그대로 사용)
    w_t.copy_(W_gpu[i])
    enc = tok(prompt_cache[i], return_tensors="pt", add_special_tokens=False).to(model.device)
    L = enc["input_ids"].shape[1]
    g = model.generate(**enc, max_new_tokens=48, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(g[0,L:], skip_special_tokens=True).strip()

def run_block(layers, coef, idxs, tag):
    detach()
    for l in layers: _hooks.append(model.model.layers[l].register_forward_hook(make_hook(l)))
    _cfg["coef"], _cfg["layers"] = coef, layers
    out=[]
    for i in tqdm(idxs, desc=tag, leave=False):
        out.append(("retain",int(i),tag.split()[0],coef, gen_txt(int(i))))
    return out

for lyr in model.model.layers: lyr._forward_hooks.clear()
_hooks.clear()

rows=[]; t0=time.time()
rows += run_block([14,15,16,17,18], 0.0, ret_idx, "base")     # base
for bn,blk in BLOCKS_R.items():
    for c in RET_C:
        rows += run_block(blk, c, ret_idx, f"{bn} {c}")
detach()
print(f"총 {time.time()-t0:.0f}s, {len(rows)}건")

gen_df = pd.DataFrame(rows, columns=["dir","idx","block","coef","text"])
gen_df.to_csv(f"{P9}/retain_sweep_gen.csv", index=False)
print("저장:", f"{P9}/retain_sweep_gen.csv")

base:   0%|          | 0/378 [00:00<?, ?it/s]

mid1 -0.3:   0%|          | 0/378 [00:00<?, ?it/s]

mid1 -0.5:   0%|          | 0/378 [00:00<?, ?it/s]

mid1 -0.7:   0%|          | 0/378 [00:00<?, ?it/s]

mid1 -1.0:   0%|          | 0/378 [00:00<?, ?it/s]

...

back -0.3:   0%|          | 0/378 [00:00<?, ?it/s]

back -0.5:   0%|          | 0/378 [00:00<?, ?it/s]

back -0.7:   0%|          | 0/378 [00:00<?, ?it/s]

back -1.0:   0%|          | 0/378 [00:00<?, ?it/s]

총 7269s, 4914건
저장: /content/drive/MyDrive/Grad_share/data_share/P9/retain_sweep_gen.csv


In [ ]:
def classify(a, gold, wrong):
    gm = match_gold(a, json.loads(gold) if str(gold).startswith("[") else [gold])
    wm = match_gold(a, [wrong]) if str(wrong).strip() else False
    if collapsed(a): return "other"
    if REFUSAL_PAT.search(a): return "abstain"
    if gm: return "retain"
    if wm: return "accept"
    return "other"

m = pairs9.set_index(pairs9.index)[["gold","wrong_answer","question"]]
gen_df["lab"] = [classify(t, m.gold[i], m.wrong_answer[i]) for t,i in zip(gen_df.text, gen_df.idx)]

base = gen_df[gen_df.block=="base"].set_index("idx")
print("baseline:", base.lab.value_counts().to_dict())
for bn in ["mid1","signal","back"]:
    print(f"\n[{bn}]")
    for c in [-0.3,-0.5,-0.7,-1.0]:
        sub = gen_df[(gen_df.block==bn)&(gen_df.coef==c)].set_index("idx")
        vc = sub.lab.value_counts()
        f_acc = ((base.lab=="retain")&(sub.lab=="accept")).sum()
        f_oth = ((base.lab=="retain")&(sub.lab=="other")).sum()
        print(f"  c{c}: retain {vc.get('retain',0):3} accept {vc.get('accept',0):3} "
              f"other {vc.get('other',0):3} | retain→accept {f_acc}  retain→other {f_oth}")

baseline: {'retain': 173, 'accept': 156, 'other': 48, 'abstain': 1}

[mid1]
  c-0.3: retain 187 accept 129 other  60 | retain→accept 14  retain→other 5
  c-0.5: retain 186 accept 120 other  68 | retain→accept 18  retain→other 14
  c-0.7: retain 192 accept 117 other  64 | retain→accept 21  retain→other 13
  c-1.0: retain 182 accept 106 other  82 | retain→accept 18  retain→other 18

[signal]
  c-0.3: retain 182 accept 125 other  69 | retain→accept 19  retain→other 12
  c-0.5: retain 182 accept 111 other  78 | retain→accept 16  retain→other 18
  c-0.7: retain 183 accept  97 other  93 | retain→accept 17  retain→other 24
  c-1.0: retain 131 accept  87 other 153 | retain→accept 31  retain→other 47

[back]
  c-0.3: retain 200 accept 121 other  56 | retain→accept 16  retain→other 5
  c-0.5: retain 191 accept 119 other  61 | retain→accept 22  retain→other 13
  c-0.7: retain 174 accept 113 other  83 | retain→accept 25  retain→other 23
  c-1.0: retain 126 accept 121 other 122 | retain→accept 41  

In [ ]:
# retain 유도의 정확한 지표: classify 라벨 기준 (retain 수 증감 + 전이 분해)
base = gen_df[gen_df.block=="base"].set_index("idx")
base_ret = (base.lab=="retain")
qs = pairs9.loc[base.index, "question"].values

def lab_bootstrap(b_ret, c_ret, groups, n=2000, seed=0):
    rng=np.random.default_rng(seed); uq=np.unique(groups)
    idx={q:np.where(groups==q)[0] for q in uq}; nets=[]
    for _ in range(n):
        rows=np.concatenate([idx[q] for q in rng.choice(uq,len(uq),replace=True)])
        nets.append(((~b_ret[rows])&(c_ret[rows])).sum()-((b_ret[rows])&(~c_ret[rows])).sum())
    return np.percentile(nets,2.5), np.percentile(nets,97.5)

print(f"baseline retain 수: {base_ret.sum()}\n")
for bn in ["mid1","signal","back"]:
    print(f"[{bn}]")
    for c in [-0.3,-0.5,-0.7,-1.0]:
        sub = gen_df[(gen_df.block==bn)&(gen_df.coef==c)].set_index("idx").loc[base.index]
        c_ret = (sub.lab=="retain").values
        gain = ((~base_ret.values)&c_ret).sum()   # 비retain→retain
        loss = ((base_ret.values)&(~c_ret)).sum()  # retain→비retain
        net = gain-loss
        lo,hi = lab_bootstrap(base_ret.values, c_ret, qs)
        sig = "유의(+)" if lo>0 else "유의(−)" if hi<0 else "불명확"
        print(f"  c{c}: retain수 {c_ret.sum():3} (net {net:+4} [{lo:.0f},{hi:.0f}] {sig}) "
              f"| ↑{gain} ↓{loss}")
    print()

baseline retain 수: 173

[mid1]
  c-0.3: retain수 187 (net  +14 [0,28] 불명확) | ↑34 ↓20
  c-0.5: retain수 186 (net  +13 [-4,30] 불명확) | ↑47 ↓34
  c-0.7: retain수 192 (net  +19 [1,37] 유의(+)) | ↑55 ↓36
  c-1.0: retain수 182 (net   +9 [-9,28] 불명확) | ↑49 ↓40

[signal]
  c-0.3: retain수 182 (net   +9 [-7,26] 불명확) | ↑41 ↓32
  c-0.5: retain수 182 (net   +9 [-8,26] 불명확) | ↑46 ↓37
  c-0.7: retain수 183 (net  +10 [-9,29] 불명확) | ↑54 ↓44
  c-1.0: retain수 131 (net  -42 [-64,-19] 유의(−)) | ↑42 ↓84

[back]
  c-0.3: retain수 200 (net  +27 [12,43] 유의(+)) | ↑48 ↓21
  c-0.5: retain수 191 (net  +18 [-1,37] 불명확) | ↑55 ↓37
  c-0.7: retain수 174 (net   +1 [-19,22] 불명확) | ↑53 ↓52
  c-1.0: retain수 126 (net  -47 [-68,-26] 유의(−)) | ↑37 ↓84



In [ ]:
# fold 겹침 검증: 각 행의 w가 자기 질문 없는 fold에서 왔는지
gkf = GroupKFold(5)
leak = 0
for tr,te in gkf.split(np.arange(len(pairs9)), groups=pairs9.question.values):
    q_tr = set(pairs9.question.values[tr]); q_te = set(pairs9.question.values[te])
    leak += len(q_tr & q_te)
print("train/test 질문 겹침:", leak, "(0이어야 누수 없음)")

train/test 질문 겹침: 0 (0이어야 누수 없음)


In [ ]:
import glob
print([f.split('/')[-1] for f in glob.glob(f"{P9}/*strat*")+glob.glob(f"{P9}/*sv*")+glob.glob(f"{P9}/hid*")])

['p8s_behavior.csv', 'steer_sweep.csv', 'hid_sv_p9prompt.npy', 'hid_sv_p9prompt_ids.npy', 'sv_layer_ckpt.npz', 'retain_sweep_gen.csv', 'hid_p8s.npy', 'hid_p8s_ids.npy', 'hid_sv_p9prompt.npy', 'hid_sv_p9prompt_ids.npy']


In [ ]:
HSV = np.load(f"{P9}/hid_sv_p9prompt.npy").astype(np.float32)
sv_ids = np.load(f"{P9}/hid_sv_p9prompt_ids.npy", allow_pickle=True)
print("HSV:", HSV.shape, "| ids:", sv_ids.shape, "| 예시:", sv_ids[:2])

# strat_v2 메타 로드 (behavior/dataset 컬럼 확인)
import glob
meta_f = glob.glob(f"{DATA}/*strat*")+glob.glob(f"{DATA}/*sv*")+glob.glob(f"{DATA}/*behavior*")
print("메타 후보:", [f.split('/')[-1] for f in meta_f])

HSV: (1916, 131072) | ids: (1916,) | 예시: ['clasheval_1902' 'clasheval_2491']
메타 후보: ['p8_pairs.csv', 'p8s_pairs.csv']


In [ ]:
import glob
cands = glob.glob("/content/drive/MyDrive/Grad_share/**/*strat*", recursive=True) \
      + glob.glob("/content/drive/MyDrive/Grad_share/**/*behavior*", recursive=True) \
      + glob.glob("/content/drive/MyDrive/Grad_share/**/*.parquet", recursive=True)
for f in sorted(set(cands)): print(f)

/content/drive/MyDrive/Grad_share/data_share/P6/attention_features.parquet
/content/drive/MyDrive/Grad_share/data_share/P6/conflict_pool_kstate.parquet
/content/drive/MyDrive/Grad_share/data_share/P6/strat_final_inst.parquet
/content/drive/MyDrive/Grad_share/data_share/P6/그 이외 데이터(필수X 필요시 사용)/conflict_pool_raw.parquet
/content/drive/MyDrive/Grad_share/data_share/P6/그 이외 데이터(필수X 필요시 사용)/conflict_pool_with_candidates.parquet
/content/drive/MyDrive/Grad_share/data_share/P6/그 이외 데이터(필수X 필요시 사용)/strat_final.parquet
/content/drive/MyDrive/Grad_share/data_share/P6/그 이외 데이터(필수X 필요시 사용)/strat_sample.parquet
/content/drive/MyDrive/Grad_share/data_share/P6/그 이외 데이터(필수X 필요시 사용)/strat_truncated.parquet
/content/drive/MyDrive/Grad_share/data_share/P9/p8s_behavior.csv
/content/drive/MyDrive/Grad_share/data_share/P9/w_behavior.npy
/content/drive/MyDrive/Grad_share/data_share/P9/w_behavior_na.npy
/content/drive/MyDrive/Grad

In [ ]:
sv = pd.read_parquet("/content/drive/MyDrive/Grad_share/data_share/P6/strat_final_inst.parquet")
print("shape:", sv.shape)
print("컬럼:", list(sv.columns))
# HSV ids와 매칭되는 id 컬럼 찾기
sv_ids = np.load(f"{P9}/hid_sv_p9prompt_ids.npy", allow_pickle=True)
for c in sv.columns:
    v = sv[c].astype(str).values
    if set(map(str, sv_ids[:20])) <= set(v):
        print(f"★ id 컬럼: {c} | 순서일치: {np.array_equal(sv[c].astype(str).values, sv_ids.astype(str))}")
print("dataset 분포:", sv['dataset'].value_counts().to_dict() if 'dataset' in sv else "dataset 컬럼 없음")
print("behavior 있나:", 'behavior' in sv.columns, "| wrong_answer 있나:", 'wrong_answer' in sv.columns)

shape: (2000, 32)
컬럼: ['sample_id', 'dataset_id', 'fact_nature', 'construction_operator', 'reasoning_structure', 'provenance', 'desired_action', 'world_truth', 'context_internal_validity', 'benchmark_target', 'question', 'context', 'parametric_answer', 'contextual_answer', 'relation_type', 'answer_type', 'hop_count', 'domain', 'mod_degree_bin', 'num_edits', 's_pop', 'mulan_date', 'answer_candidates', 'cb_generation', 'K_state3', 'K', 'n_known', 'instability', 'K_final', 'prompt_tok_len', 'context_trunc', 'prompt_tok_len_trunc']
dataset 분포: dataset 컬럼 없음
behavior 있나: False | wrong_answer 있나: False


In [ ]:
import glob
f = glob.glob("/content/drive/MyDrive/**/strat_v2_behavior.parquet", recursive=True) \
  + glob.glob("/content/**/strat_v2_behavior.parquet", recursive=True)
print("찾음:", f if f else "없음 — Drive에 없음")

찾음: ['/content/drive/MyDrive/Grad_share/data_share/P6/strat_v2_behavior.parquet', '/content/drive/MyDrive/Grad_share/data_share/P6/strat_v2_behavior.parquet']


In [ ]:
SV_META = "/content/drive/MyDrive/Grad_share/data_share/P6/strat_v2_behavior.parquet"
sv = pd.read_parquet(SV_META)
HSV = np.load(f"{P9}/hid_sv_p9prompt.npy").astype(np.float32)
sv_ids = np.load(f"{P9}/hid_sv_p9prompt_ids.npy", allow_pickle=True)

print("sv:", sv.shape, "| HSV:", HSV.shape, "| ids:", sv_ids.shape)
print("컬럼:", list(sv.columns))

# id 컬럼 + 정렬 확인
for c in sv.columns:
    if set(map(str, sv_ids[:20])) <= set(sv[c].astype(str).values):
        print(f"★ id컬럼 '{c}' | 순서일치:", np.array_equal(sv[c].astype(str).values, sv_ids.astype(str)))
        break

# 핵심 컬럼 확인
for col in ['dataset','behavior','gold','wrong_answer','kside','ctx_label','question','ctx_text','context']:
    if col in sv.columns: print(f"  {col}: 있음 | 예시 {sv[col].iloc[0]!r:.40}")
print("\nbehavior 분포:", sv.behavior.value_counts().to_dict() if 'behavior' in sv else "없음")
print("dataset 분포:", sv.dataset.value_counts().to_dict() if 'dataset' in sv else "없음")

sv: (1916, 43) | HSV: (1916, 131072) | ids: (1916,)
컬럼: ['sample_id', 'dataset_id', 'fact_nature', 'construction_operator', 'reasoning_structure', 'provenance', 'desired_action', 'world_truth', 'context_internal_validity', 'benchmark_target', 'question', 'context', 'parametric_answer', 'contextual_answer', 'relation_type', 'answer_type', 'hop_count', 'domain', 'mod_degree_bin', 'num_edits', 's_pop', 'mulan_date', 'answer_candidates', 'cb_generation', 'K_state3', 'K', 'ans_ok', 'context_trunc', 'ans_preserved', 'prompt_len', 'cb_samples', 'n_known5', 'instability', 'refusal_rate', 'greedy_refusal', 'is_abstain_group', 'cb_samples_str', 'greedy_known_v2', 'open_book_gen', 'n_steps', 'behavior', 'no_conflict', 'hedged']
★ id컬럼 'sample_id' | 순서일치: True
  behavior: 있음 | 예시 'both'
  question: 있음 | 예시 "What year was Phil Knight's memoir, 'Sh
  context: 있음 | 예시 'Philip Hampson Knight (born February 24

behavior 분포: {'accepted_context': 846, 'both': 352, 'retained_parametric': 323, 'other': 264

In [ ]:
# 1) 벤치마크 소스 확인
print("sample_id 접두어:", pd.Series([s.split('_')[0] for s in sv.sample_id]).value_counts().to_dict())
print("dataset_id 분포:", sv.dataset_id.value_counts().to_dict())

# 2) K_state / desired_action로 retain 대상(K+, 파라메트릭이 정답) 식별 가능한지
print("\nK 분포:", sv.K.value_counts().to_dict() if 'K' in sv else "없음")
print("desired_action 분포:", sv.desired_action.value_counts().to_dict())
print("world_truth 분포:", sv.world_truth.value_counts().to_dict())

# 3) gold/wrong 대체 컬럼 미리보기
print("\nparametric_answer 예시:", sv.parametric_answer.iloc[0])
print("contextual_answer 예시:", sv.contextual_answer.iloc[0])

sample_id 접두어: {'clasheval': 480, 'confiqa': 480, 'dynamicqa': 479, 'nqswap': 477}
dataset_id 분포: {'clasheval': 480, 'confiqa_qa': 480, 'dynamicqa': 479, 'nqswap': 477}

K 분포: {'K-': 968, 'K+': 948}
desired_action 분포: {'accept': 957, 'reject': 480, 'needs_verification': 479}
world_truth 분포: {'false': 1670, 'outdated': 200, 'viewpoint_dependent': 46}

parametric_answer 예시: 2016
contextual_answer 예시: 1961


In [ ]:
sv["bench"] = sv.sample_id.str.split("_").str[0]
sv["beh3"] = sv.behavior.map({"retained_parametric":"retain","accepted_context":"accept",
                               "both":"retain","other":"other","abstain":"abstain"})
for b in ["confiqa","nqswap","clasheval","dynamicqa"]:
    sub = sv[(sv.bench==b)&(sv.K=="K+")]
    print(f"{b}: K+ {len(sub)}행 | retain {int((sub.beh3=='retain').sum())} "
          f"accept {int((sub.beh3=='accept').sum())} baseline수={int((sub.beh3=='retain').sum())}")

confiqa: K+ 244행 | retain 168 accept 49 baseline수=168
nqswap: K+ 247행 | retain 136 accept 70 baseline수=136
clasheval: K+ 222행 | retain 70 accept 124 baseline수=70
dynamicqa: K+ 235행 | retain 131 accept 79 baseline수=131


In [ ]:
sv["bench"] = sv.sample_id.str.split("_").str[0]
sv["beh3"] = sv.behavior.map({"retained_parametric":"retain","accepted_context":"accept",
                               "both":"retain","other":"other","abstain":"abstain"})
sv["gold_p"]  = sv.parametric_answer.astype(str)   # retain 정답
sv["wrong_c"] = sv.contextual_answer.astype(str)   # accept 시 따라가는 답

def build_prompt_sv(row):
    return tok.apply_chat_template(
        [{"role":"user","content":f"{row.context}\n\nQuestion: {row.question}\nAnswer:"}],
        add_generation_prompt=True, tokenize=False)

def classify_sv(a, gold, wrong):
    gm = match_gold(a, [gold]); wm = match_gold(a, [wrong]) if str(wrong).strip() else False
    if collapsed(a): return "other"
    if REFUSAL_PAT.search(a): return "abstain"
    if gm: return "retain"
    if wm: return "accept"
    return "other"
print("준비 완료 | HSV:", HSV.shape)

준비 완료 | HSV: (1916, 131072)


In [ ]:
def build_woof_bench(bench):
    idx = sv[sv.bench==bench].index.values
    Hb = HSV[idx]                                    # (Nb, 32*4096)
    lab = sv.loc[idx,"beh3"].values
    q   = sv.loc[idx,"question"].values
    m_ar = np.isin(lab, ["accept","retain"])
    W = np.zeros((len(idx),32,4096), np.float32)
    S = np.zeros((len(idx),32), np.float32)
    from sklearn.model_selection import GroupKFold
    for tr,te in GroupKFold(5).split(np.arange(len(idx)), groups=q):
        gi = tr[m_ar[tr]]
        Wtr = diff_of_means(Hb[gi], lab[gi]=="accept", lab[gi]=="retain")
        for j in te: W[j]=Wtr
        for L in range(1,33):
            S[te,L-1] = (layer(Hb[gi],L) @ Wtr[L-1]).std()
    return idx, W, np.median(S,axis=0).astype(np.float32)
print("함수 준비")

함수 준비


In [ ]:
BLOCKS_R = {"mid1":[11,12,13,14,15],"signal":[14,15,16,17,18],"back":[17,18,19,20,21],"front":[8,9,10,11,12]}
RET_C = [-0.3,-0.5,-0.7,-1.0]

def sweep_bench(bench):
    idx, W, sig = build_woof_bench(bench)
    sig_t.copy_(torch.tensor(sig, device=model.device, dtype=torch.bfloat16))
    ret_idx = [j for j in range(len(idx)) if sv.loc[idx[j],"K"]=="K+"]   # K+ 셀
    W_gpu = {j: torch.tensor(W[j], device=model.device, dtype=torch.bfloat16) for j in ret_idx}
    pc = {j: build_prompt_sv(sv.loc[idx[j]]) for j in ret_idx}

    @torch.no_grad()
    def gen(j):
        w_t.copy_(W_gpu[j])
        enc=tok(pc[j],return_tensors="pt",add_special_tokens=False).to(model.device)
        L=enc["input_ids"].shape[1]
        g=model.generate(**enc,max_new_tokens=48,do_sample=False,pad_token_id=tok.pad_token_id)
        return tok.decode(g[0,L:],skip_special_tokens=True).strip()

    rows=[]
    for lyr in model.model.layers: lyr._forward_hooks.clear()
    _hooks.clear()
    # base: 훅 없이 (최속)
    detach()
    for j in tqdm(ret_idx, desc=f"{bench}-base", leave=False):
        rows.append((bench,int(idx[j]),"base",0.0, gen(j)))
    # blocks: 블록당 훅 1회
    for bn,blk in BLOCKS_R.items():
        detach()
        for l in blk: _hooks.append(model.model.layers[l].register_forward_hook(make_hook(l)))
        for c in RET_C:
            _cfg["coef"],_cfg["layers"]=c,blk
            for j in tqdm(ret_idx, desc=f"{bench}-{bn}{c}", leave=False):
                rows.append((bench,int(idx[j]),bn,c, gen(j)))
    detach()
    return rows

all_rows=[]
for b in ["confiqa","nqswap"]:      # 우선 2개
    all_rows += sweep_bench(b)
gen_sv = pd.DataFrame(all_rows, columns=["bench","sid","block","coef","text"])
gen_sv.to_csv(f"{P9}/bench_retain_sweep.csv", index=False)
print("저장:", len(gen_sv), "→", f"{P9}/bench_retain_sweep.csv")

confiqa-base:   0%|          | 0/244 [00:00<?, ?it/s]

confiqa-mid1-0.3:   0%|          | 0/244 [00:00<?, ?it/s]

confiqa-mid1-0.5:   0%|          | 0/244 [00:00<?, ?it/s]

confiqa-mid1-0.7:   0%|          | 0/244 [00:00<?, ?it/s]

confiqa-mid1-1.0:   0%|          | 0/244 [00:00<?, ?it/s]

...

nqswap-front-0.3:   0%|          | 0/247 [00:00<?, ?it/s]

nqswap-front-0.5:   0%|          | 0/247 [00:00<?, ?it/s]

nqswap-front-0.7:   0%|          | 0/247 [00:00<?, ?it/s]

nqswap-front-1.0:   0%|          | 0/247 [00:00<?, ?it/s]

저장: 8347 → /content/drive/MyDrive/Grad_share/data_share/P9/bench_retain_sweep.csv


In [ ]:
gen_sv["lab"] = [classify_sv(t, sv.loc[s,"gold_p"], sv.loc[s,"wrong_c"]) for t,s in zip(gen_sv.text,gen_sv.sid)]
for b in ["confiqa","nqswap"]:
    g=gen_sv[gen_sv.bench==b]; base=g[g.block=="base"].set_index("sid")
    br=(base.lab=="retain"); qs=sv.loc[base.index,"question"].values
    print(f"\n=== {b} (baseline retain {br.sum()}/{len(base)}) ===")
    for bn in BLOCKS_R:
        best=None
        for c in RET_C:
            sub=g[(g.block==bn)&(g.coef==c)].set_index("sid").loc[base.index]
            cr=(sub.lab=="retain").values
            net=((~br.values)&cr).sum()-((br.values)&(~cr)).sum()
            lo,hi=lab_bootstrap(br.values,cr,qs)
            if best is None or net>best[1]: best=(c,net,(int(lo),int(hi)))
        print(f"  {bn:7} 최적 c{best[0]}: net {best[1]:+3} CI{best[2]}")


=== confiqa (baseline retain 102/244) ===
  mid1    최적 c-0.7: net +11 CI(-2, 24)
  signal  최적 c-0.7: net +16 CI(2, 29)
  back    최적 c-0.3: net  +7 CI(-3, 18)
  front   최적 c-0.3: net  +0 CI(-8, 7)

=== nqswap (baseline retain 82/247) ===
  mid1    최적 c-1.0: net +59 CI(43, 76)
  signal  최적 c-0.7: net +50 CI(35, 66)
  back    최적 c-0.7: net +46 CI(30, 62)
  front   최적 c-1.0: net +37 CI(23, 51)


In [ ]:
import os
CKPT = f"{P9}/bench_retain_sweep2.csv"   # dynamicqa+clasheval
done = set()
if os.path.exists(CKPT):
    d = pd.read_csv(CKPT); done = set(zip(d.bench, d.block, d.coef.round(1)))
    print("이미 완료:", len(done), "조건")

def sweep_bench_ckpt(bench):
    idx, W, sig = build_woof_bench(bench)
    sig_t.copy_(torch.tensor(sig, device=model.device, dtype=torch.bfloat16))
    ret_idx = [j for j in range(len(idx)) if sv.loc[idx[j],"K"]=="K+"]
    W_gpu = {j: torch.tensor(W[j], device=model.device, dtype=torch.bfloat16) for j in ret_idx}
    pc = {j: build_prompt_sv(sv.loc[idx[j]]) for j in ret_idx}

    @torch.no_grad()
    def gen(j):
        w_t.copy_(W_gpu[j])
        enc=tok(pc[j],return_tensors="pt",add_special_tokens=False).to(model.device)
        L=enc["input_ids"].shape[1]
        g=model.generate(**enc,max_new_tokens=48,do_sample=False,pad_token_id=tok.pad_token_id)
        return tok.decode(g[0,L:],skip_special_tokens=True).strip()

    def flush(chunk):
        pd.DataFrame(chunk,columns=["bench","sid","block","coef","text"]).to_csv(
            CKPT, mode="a", header=not os.path.exists(CKPT), index=False)

    for lyr in model.model.layers: lyr._forward_hooks.clear()
    _hooks.clear()
    # base
    if (bench,"base",0.0) not in done:
        detach(); chunk=[]
        for j in tqdm(ret_idx, desc=f"{bench}-base", leave=False):
            chunk.append((bench,int(idx[j]),"base",0.0, gen(j)))
        flush(chunk)
    # blocks
    for bn,blk in BLOCKS_R.items():
        detach()
        for l in blk: _hooks.append(model.model.layers[l].register_forward_hook(make_hook(l)))
        for c in RET_C:
            if (bench,bn,round(c,1)) in done: continue
            _cfg["coef"],_cfg["layers"]=c,blk
            chunk=[]
            for j in tqdm(ret_idx, desc=f"{bench}-{bn}{c}", leave=False):
                chunk.append((bench,int(idx[j]),bn,c, gen(j)))
            flush(chunk)          # 조건마다 저장
    detach()

for b in ["dynamicqa","clasheval"]:
    sweep_bench_ckpt(b)
print("완료 →", CKPT)

dynamicqa-base:   0%|          | 0/235 [00:00<?, ?it/s]

dynamicqa-mid1-0.3:   0%|          | 0/235 [00:00<?, ?it/s]

dynamicqa-mid1-0.5:   0%|          | 0/235 [00:00<?, ?it/s]

dynamicqa-mid1-0.7:   0%|          | 0/235 [00:00<?, ?it/s]

dynamicqa-mid1-1.0:   0%|          | 0/235 [00:00<?, ?it/s]

...

clasheval-front-0.3:   0%|          | 0/222 [00:00<?, ?it/s]

clasheval-front-0.5:   0%|          | 0/222 [00:00<?, ?it/s]

clasheval-front-0.7:   0%|          | 0/222 [00:00<?, ?it/s]

clasheval-front-1.0:   0%|          | 0/222 [00:00<?, ?it/s]

완료 → /content/drive/MyDrive/Grad_share/data_share/P9/bench_retain_sweep2.csv


In [ ]:
g_all = pd.read_csv(CKPT)
g_all["lab"] = [classify_sv(t, sv.loc[s,"gold_p"], sv.loc[s,"wrong_c"]) for t,s in zip(g_all.text,g_all.sid)]
for b in ["dynamicqa","clasheval"]:
    g=g_all[g_all.bench==b]; base=g[g.block=="base"].set_index("sid")
    br=(base.lab=="retain"); qs=sv.loc[base.index,"question"].values
    print(f"\n=== {b} (baseline retain {br.sum()}/{len(base)}) ===")
    for bn in BLOCKS_R:
        best=None
        for c in RET_C:
            sub=g[(g.block==bn)&(g.coef==c)].set_index("sid").loc[base.index]
            cr=(sub.lab=="retain").values
            net=((~br.values)&cr).sum()-((br.values)&(~cr)).sum()
            lo,hi=lab_bootstrap(br.values,cr,qs)
            if best is None or net>best[1]: best=(c,net,(int(lo),int(hi)))
        print(f"  {bn:7} 최적 c{best[0]}: net {best[1]:+3} CI{best[2]}")


=== dynamicqa (baseline retain 80/235) ===
  mid1    최적 c-0.3: net +23 CI(10, 36)
  signal  최적 c-0.3: net +17 CI(4, 31)
  back    최적 c-0.3: net  +7 CI(-6, 20)
  front   최적 c-0.3: net +13 CI(5, 21)

=== clasheval (baseline retain 60/222) ===
  mid1    최적 c-0.7: net  +8 CI(-2, 19)
  signal  최적 c-0.3: net  +5 CI(-3, 14)
  back    최적 c-0.5: net +11 CI(2, 21)
  front   최적 c-0.3: net  +5 CI(-2, 13)


In [ ]:
import time
detach()
for l in [14,15,16,17,18]: _hooks.append(model.model.layers[l].register_forward_hook(make_hook(l)))
_cfg["coef"], _cfg["layers"] = 0.0, [14,15,16,17,18]
i=int(ret_idx[0])

t=time.time(); w_t.copy_(W_gpu[i]); torch.cuda.synchronize(); print("copy:", round((time.time()-t)*1000,1),"ms")
t=time.time(); enc=tok(prompt_cache[i], return_tensors="pt", add_special_tokens=False).to(model.device); torch.cuda.synchronize(); print("tok:", round((time.time()-t)*1000,1),"ms")
t=time.time(); g=model.generate(**enc, max_new_tokens=48, do_sample=False, pad_token_id=tok.pad_token_id); torch.cuda.synchronize(); print("gen:", round((time.time()-t)*1000,1),"ms | tok", g.shape[1]-enc['input_ids'].shape[1])
detach()

copy: 0.6 ms
tok: 1.9 ms
gen: 81.8 ms | tok 2


In [ ]:
def classify(a, gold, wrong):
    gm = match_gold(a, json.loads(gold) if str(gold).startswith("[") else [gold])
    wm = match_gold(a, [wrong]) if str(wrong).strip() else False
    if collapsed(a): return "other"
    if REFUSAL_PAT.search(a): return "abstain"
    if gm: return "retain"
    if wm: return "accept"
    return "other"

m = pairs9[["gold","wrong_answer","question"]]
gen_df["lab"] = [classify(t, m.gold[i], m.wrong_answer[i]) for t,i in zip(gen_df.text, gen_df.idx)]
gen_df["q"]   = [m.question[i] for i in gen_df.idx]

base = gen_df[gen_df.block=="base"].set_index("idx")
print("baseline:", base.lab.value_counts().to_dict())
for bn in BLOCKS_R:
    print(f"\n[{bn}]")
    for c in RET_C:
        sub = gen_df[(gen_df.block==bn)&(gen_df.coef==c)]
        vc = sub.lab.value_counts()
        # base 대비 retain 유지가 어디로 샜는지
        merged = sub.set_index("idx")
        flip_acc = ((base.lab=="retain") & (merged.lab=="accept")).sum()
        flip_oth = ((base.lab=="retain") & (merged.lab=="other")).sum()
        print(f"  c{c}: retain {vc.get('retain',0):3} accept {vc.get('accept',0):3} "
              f"other {vc.get('other',0):3} | retain→accept {flip_acc} retain→other {flip_oth}")